In [2]:
from pathlib import Path

import pandas as pd
# ---------------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

LOAN_FILE = RAW_DATA_DIR / "loan_portfolio.csv"
# ---------------------------------------------------------------------------
# Validate project structure
# ---------------------------------------------------------------------------
required_directories = (
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
)

for directory in required_directories:
    directory.mkdir(parents=True, exist_ok=True)

if not LOAN_FILE.exists():
    raise FileNotFoundError(
        f"Required dataset was not found: {LOAN_FILE}"
    )
# ---------------------------------------------------------------------------
# Load raw dataset
# ---------------------------------------------------------------------------
loans = pd.read_csv(
    LOAN_FILE,
    low_memory=False,
)
print(f"Dataset loaded successfully: {LOAN_FILE.name}")
print(f"Rows: {loans.shape[0]:,}")
print(f"Columns: {loans.shape[1]:,}")

Dataset loaded successfully: loan_portfolio.csv
Rows: 50,000
Columns: 24


In [3]:
# ---------------------------------------------------------------------------
# Initial data inspection
# ---------------------------------------------------------------------------
display(loans.head())

print("\nData types:")
display(loans.dtypes)

print("\nMissing values:")
display(
    loans.isna()
         .sum()
         .sort_values(ascending=False)
         .to_frame("missing_count")
)

,loan_id,origination_date,maturity_date,maturity_months,sector,loan_type,collateral,initial_rating,credit_score,ead,...,pd_annual,lgd,el,unexpected_loss,rwa,defaulted,default_date,survival_months,recovery_rate,loss_given_default
0,L000001,2017-10-01,2019-10-01,24,Technology,mortgage,secured,B,704,1035611.99,...,0.041076,0.7858,33426.80,161508.03,442905.14,0,NaN,24,NaN,NaN
1,L000002,2016-06-01,2020-06-01,48,Healthcare,term_loan,unsecured,A,702,800786.92,...,0.000887,0.7034,499.82,16771.62,6622.66,0,NaN,48,NaN,NaN
2,L000003,2023-08-01,2024-12-31,36,Real_Estate,bond,secured,BBB,699,4327864.12,...,0.002622,0.5503,6243.54,121781.72,82726.94,0,NaN,36,NaN,NaN
3,L000004,2016-04-01,2024-12-31,120,Real_Estate,term_loan,secured,B,641,3810461.19,...,0.041679,0.3541,56237.33,269661.85,745144.64,0,NaN,120,NaN,NaN
4,L000005,2015-10-01,2019-10-01,48,Energy,bond,unsecured,CCC,597,2925538.53,...,0.136071,0.6576,261778.69,659614.05,3468567.58,0,NaN,48,NaN,NaN



Data types:


loan_id                   str
origination_date          str
maturity_date             str
maturity_months         int64
sector                    str
loan_type                 str
collateral                str
initial_rating            str
credit_score            int64
ead                   float64
coupon_rate           float64
leverage              float64
interest_coverage     float64
debt_to_equity        float64
pd_annual             float64
lgd                   float64
el                    float64
unexpected_loss       float64
rwa                   float64
defaulted               int64
default_date              str
survival_months         int64
recovery_rate         float64
loss_given_default    float64
dtype: object


Missing values:


,missing_count
default_date,43050
recovery_rate,43050
loss_given_default,43050
loan_id,0
sector,0
origination_date,0
maturity_date,0
maturity_months,0
initial_rating,0
collateral,0


In [4]:
# ---------------------------------------------------------------------------
# Data-quality checks
# ---------------------------------------------------------------------------
duplicate_count = loans.duplicated().sum()

print(f"Duplicate rows: {duplicate_count:,}")

if duplicate_count > 0:
    print("WARNING: Duplicate rows require investigation.")
else:
    print("No exact duplicate rows detected.")

Duplicate rows: 0
No exact duplicate rows detected.


In [5]:
loans = pd.read_csv(LOAN_FILE)

In [7]:
clean_file = PROCESSED_DATA_DIR / "loan_portfolio_clean.csv"
loans.to_csv(
    clean_file,
    index=False,
)

In [8]:
# ---------------------------------------------------------------------------
# Dataset inventory
# ---------------------------------------------------------------------------

DATASETS = {
    "loan_portfolio": RAW_DATA_DIR / "loan_portfolio.csv",
    "credit_ratings": RAW_DATA_DIR / "credit_ratings.csv",
    "macro_stress_scenarios": RAW_DATA_DIR / "macro_stress_scenarios.csv",
    "portfolio_metrics": RAW_DATA_DIR / "portfolio_metrics.csv",
    "vintage_analysis": RAW_DATA_DIR / "vintage_analysis.csv",
}


def validate_dataset_files(dataset_paths: dict[str, Path]) -> None:
    """Validate that all expected source files are available."""
    
    missing_files = [
        path for path in dataset_paths.values()
        if not path.exists()
    ]

    if missing_files:
        missing = "\n".join(f"- {path}" for path in missing_files)
        raise FileNotFoundError(
            f"The following required dataset files are missing:\n{missing}"
        )


validate_dataset_files(DATASETS)

print(f"Validated {len(DATASETS)} source datasets.")

Validated 5 source datasets.


In [9]:
# ---------------------------------------------------------------------------
# Load dataset metadata
# ---------------------------------------------------------------------------

dataset_inventory = []

for dataset_name, dataset_path in DATASETS.items():
    dataset_inventory.append(
        {
            "dataset": dataset_name,
            "file_name": dataset_path.name,
            "file_size_mb": round(
                dataset_path.stat().st_size / (1024 ** 2), 2
            ),
        }
    )

dataset_inventory = pd.DataFrame(dataset_inventory)

display(dataset_inventory)

,dataset,file_name,file_size_mb
0,loan_portfolio,loan_portfolio.csv,7.35
1,credit_ratings,credit_ratings.csv,0.61
2,macro_stress_scenarios,macro_stress_scenarios.csv,0.01
3,portfolio_metrics,portfolio_metrics.csv,0.02
4,vintage_analysis,vintage_analysis.csv,0.11


In [10]:
# ---------------------------------------------------------------------------
# Load source datasets
# ---------------------------------------------------------------------------

loan_portfolio = pd.read_csv(
    DATASETS["loan_portfolio"],
    low_memory=False,
)

credit_ratings = pd.read_csv(
    DATASETS["credit_ratings"],
    low_memory=False,
)

macro_stress_scenarios = pd.read_csv(
    DATASETS["macro_stress_scenarios"],
    low_memory=False,
)

portfolio_metrics = pd.read_csv(
    DATASETS["portfolio_metrics"],
    low_memory=False,
)

vintage_analysis = pd.read_csv(
    DATASETS["vintage_analysis"],
    low_memory=False,
)

In [11]:
# ---------------------------------------------------------------------------
# Dataset dimensions
# ---------------------------------------------------------------------------

dataset_shapes = pd.DataFrame(
    {
        "dataset": [
            "loan_portfolio",
            "credit_ratings",
            "macro_stress_scenarios",
            "portfolio_metrics",
            "vintage_analysis",
        ],
        "rows": [
            len(loan_portfolio),
            len(credit_ratings),
            len(macro_stress_scenarios),
            len(portfolio_metrics),
            len(vintage_analysis),
        ],
        "columns": [
            loan_portfolio.shape[1],
            credit_ratings.shape[1],
            macro_stress_scenarios.shape[1],
            portfolio_metrics.shape[1],
            vintage_analysis.shape[1],
        ],
    }
)

display(dataset_shapes)

,dataset,rows,columns
0,loan_portfolio,50000,24
1,credit_ratings,17939,9
2,macro_stress_scenarios,60,16
3,portfolio_metrics,120,16
4,vintage_analysis,2160,9


In [11]:
# ---------------------------------------------------------------------------
# Dataset dimensions
# ---------------------------------------------------------------------------

dataset_shapes = pd.DataFrame(
    {
        "dataset": [
            "loan_portfolio",
            "credit_ratings",
            "macro_stress_scenarios",
            "portfolio_metrics",
            "vintage_analysis",
        ],
        "rows": [
            len(loan_portfolio),
            len(credit_ratings),
            len(macro_stress_scenarios),
            len(portfolio_metrics),
            len(vintage_analysis),
        ],
        "columns": [
            loan_portfolio.shape[1],
            credit_ratings.shape[1],
            macro_stress_scenarios.shape[1],
            portfolio_metrics.shape[1],
            vintage_analysis.shape[1],
        ],
    }
)

display(dataset_shapes)

,dataset,rows,columns
0,loan_portfolio,50000,24
1,credit_ratings,17939,9
2,macro_stress_scenarios,60,16
3,portfolio_metrics,120,16
4,vintage_analysis,2160,9


In [12]:
# ---------------------------------------------------------------------------
# Data types and missing-value summary
# ---------------------------------------------------------------------------

datasets = {
    "loan_portfolio": loan_portfolio,
    "credit_ratings": credit_ratings,
    "macro_stress_scenarios": macro_stress_scenarios,
    "portfolio_metrics": portfolio_metrics,
    "vintage_analysis": vintage_analysis,
}


def profile_dataset(name: str, dataframe: pd.DataFrame) -> pd.DataFrame:
    """Create a column-level data-quality profile for one dataset."""
    
    profile = pd.DataFrame({
        "dataset": name,
        "column": dataframe.columns,
        "dtype": dataframe.dtypes.astype(str).values,
        "missing_count": dataframe.isna().sum().values,
        "missing_pct": (
            dataframe.isna().mean().mul(100).round(2).values
        ),
        "unique_values": dataframe.nunique(dropna=False).values,
    })

    return profile


dataset_profiles = pd.concat(
    [
        profile_dataset(name, dataframe)
        for name, dataframe in datasets.items()
    ],
    ignore_index=True,
)

display(dataset_profiles)

,dataset,column,dtype,missing_count,missing_pct,unique_values
0,loan_portfolio,loan_id,str,0,0.0,50000
1,loan_portfolio,origination_date,str,0,0.0,108
2,loan_portfolio,maturity_date,str,0,0.0,109
3,loan_portfolio,maturity_months,int64,0,0.0,7
4,loan_portfolio,sector,str,0,0.0,10
...,...,...,...,...,...,...
69,vintage_analysis,n_defaulted_cumulative,int64,0,0.0,280
70,vintage_analysis,cumulative_default_rate,float64,0,0.0,1614
71,vintage_analysis,marginal_default_rate,float64,0,0.0,1185
72,vintage_analysis,avg_pd_at_origination,float64,0,0.0,36


In [13]:
# ---------------------------------------------------------------------------
# Dataset-level quality summary
# ---------------------------------------------------------------------------

dataset_quality_summary = pd.DataFrame(
    [
        {
            "dataset": name,
            "rows": dataframe.shape[0],
            "columns": dataframe.shape[1],
            "duplicate_rows": dataframe.duplicated().sum(),
            "total_missing_values": int(dataframe.isna().sum().sum()),
            "columns_with_missing_values": int(
                dataframe.isna().any().sum()
            ),
        }
        for name, dataframe in datasets.items()
    ]
)

display(dataset_quality_summary)

,dataset,rows,columns,duplicate_rows,total_missing_values,columns_with_missing_values
0,loan_portfolio,50000,24,0,129150,3
1,credit_ratings,17939,9,0,0,0
2,macro_stress_scenarios,60,16,0,0,0
3,portfolio_metrics,120,16,0,0,0
4,vintage_analysis,2160,9,0,0,0


In [14]:
# ---------------------------------------------------------------------------
# Duplicate and key-integrity checks
# ---------------------------------------------------------------------------

duplicate_summary = pd.DataFrame(
    [
        {
            "dataset": name,
            "duplicate_rows": int(dataframe.duplicated().sum()),
            "duplicate_pct": round(
                dataframe.duplicated().mean() * 100, 4
            ),
        }
        for name, dataframe in datasets.items()
    ]
)

display(duplicate_summary)

,dataset,duplicate_rows,duplicate_pct
0,loan_portfolio,0,0.0
1,credit_ratings,0,0.0
2,macro_stress_scenarios,0,0.0
3,portfolio_metrics,0,0.0
4,vintage_analysis,0,0.0


In [15]:
# ---------------------------------------------------------------------------
# Candidate identifier and key columns
# ---------------------------------------------------------------------------

for name, dataframe in datasets.items():
    print(f"\n{'=' * 70}")
    print(f"{name.upper()}")
    print(f"{'=' * 70}")

    identifier_candidates = [
        column
        for column in dataframe.columns
        if any(
            keyword in column.lower()
            for keyword in ("id", "loan", "account", "customer", "portfolio")
        )
    ]

    print("Candidate identifier columns:")
    print(identifier_candidates)


LOAN_PORTFOLIO
Candidate identifier columns:
['loan_id', 'loan_type']

CREDIT_RATINGS
Candidate identifier columns:
['issuer_id']

MACRO_STRESS_SCENARIOS
Candidate identifier columns:
[]

PORTFOLIO_METRICS
Candidate identifier columns:
['n_active_loans']

VINTAGE_ANALYSIS
Candidate identifier columns:
['n_loans_originated']


In [16]:
# ---------------------------------------------------------------------------
# Identifier uniqueness checks
# ---------------------------------------------------------------------------

identifier_summary = []

for name, dataframe in datasets.items():
    identifier_candidates = [
        column
        for column in dataframe.columns
        if any(
            keyword in column.lower()
            for keyword in ("id", "loan", "account", "customer")
        )
    ]

    for column in identifier_candidates:
        identifier_summary.append(
            {
                "dataset": name,
                "column": column,
                "unique_values": int(dataframe[column].nunique(dropna=False)),
                "rows": int(len(dataframe)),
                "is_unique": bool(dataframe[column].is_unique),
                "missing_values": int(dataframe[column].isna().sum()),
            }
        )

identifier_summary = pd.DataFrame(identifier_summary)

display(identifier_summary)

,dataset,column,unique_values,rows,is_unique,missing_values
0,loan_portfolio,loan_id,50000,50000,True,0
1,loan_portfolio,loan_type,5,50000,False,0
2,credit_ratings,issuer_id,2000,17939,False,0
3,portfolio_metrics,n_active_loans,119,120,False,0
4,vintage_analysis,n_loans_originated,32,2160,False,0


In [17]:
# ---------------------------------------------------------------------------
# Column inventory by dataset
# ---------------------------------------------------------------------------

for name, dataframe in datasets.items():
    print(f"\n{'=' * 80}")
    print(f"{name.upper()}")
    print(f"{'=' * 80}")

    column_inventory = pd.DataFrame(
        {
            "column": dataframe.columns,
            "dtype": dataframe.dtypes.astype(str).values,
            "missing_count": dataframe.isna().sum().values,
            "unique_values": dataframe.nunique(dropna=False).values,
        }
    )

    display(column_inventory)


LOAN_PORTFOLIO


,column,dtype,missing_count,unique_values
0,loan_id,str,0,50000
1,origination_date,str,0,108
2,maturity_date,str,0,109
3,maturity_months,int64,0,7
4,sector,str,0,10
5,loan_type,str,0,5
6,collateral,str,0,3
7,initial_rating,str,0,7
8,credit_score,int64,0,307
9,ead,float64,0,49566



CREDIT_RATINGS


,column,dtype,missing_count,unique_values
0,issuer_id,str,0,2000
1,sector,str,0,10
2,year,int64,0,10
3,from_rating,str,0,7
4,to_rating,str,0,8
5,upgraded,int64,0,2
6,downgraded,int64,0,2
7,defaulted,int64,0,2
8,notches_moved,int64,0,13



MACRO_STRESS_SCENARIOS


,column,dtype,missing_count,unique_values
0,scenario,str,0,6
1,gdp_shock_pp,float64,0,6
2,unemp_shock_pp,float64,0,6
3,rate_shock_pp,float64,0,6
4,credit_spread_bps,int64,0,6
5,sector,str,0,10
6,base_pd,float64,0,10
7,stressed_pd,float64,0,60
8,pd_uplift_pp,float64,0,51
9,pd_multiplier,float64,0,45



PORTFOLIO_METRICS


,column,dtype,missing_count,unique_values
0,date,str,0,120
1,n_active_loans,int64,0,119
2,total_ead,float64,0,120
3,total_el,float64,0,120
4,total_rwa,float64,0,120
5,el_rate,float64,0,116
6,avg_pd,float64,0,116
7,avg_lgd,float64,0,29
8,var_99,float64,0,120
9,cvar_995,float64,0,120



VINTAGE_ANALYSIS


,column,dtype,missing_count,unique_values
0,vintage,str,0,36
1,months_on_books,int64,0,60
2,n_loans_originated,int64,0,32
3,n_active,int64,0,734
4,n_defaulted_cumulative,int64,0,280
5,cumulative_default_rate,float64,0,1614
6,marginal_default_rate,float64,0,1185
7,avg_pd_at_origination,float64,0,36
8,avg_credit_score,float64,0,23


In [18]:
# ---------------------------------------------------------------------------
# Categorical value and date-range inspection
# ---------------------------------------------------------------------------

categorical_columns = {
    "loan_portfolio": [
        "sector",
        "loan_type",
        "collateral",
        "initial_rating",
    ],
    "credit_ratings": [
        "sector",
        "from_rating",
        "to_rating",
    ],
    "macro_stress_scenarios": [
        "scenario",
        "sector",
    ],
}

for dataset_name, columns in categorical_columns.items():
    dataframe = datasets[dataset_name]

    print(f"\n{'=' * 80}")
    print(f"{dataset_name.upper()} — CATEGORICAL VALUES")
    print(f"{'=' * 80}")

    for column in columns:
        print(f"\n{column}")
        display(
            dataframe[column]
            .value_counts(dropna=False)
            .rename_axis(column)
            .reset_index(name="count")
        )


LOAN_PORTFOLIO — CATEGORICAL VALUES

sector


,sector,count
0,Energy,5132
1,Telecom,5077
2,Consumer,5043
3,Financials,5023
4,Industrials,5012
5,Healthcare,4982
6,Technology,4959
7,Utilities,4959
8,Retail,4951
9,Real_Estate,4862



loan_type


,loan_type,count
0,term_loan,17594
1,mortgage,12262
2,revolving,10158
3,bond,7458
4,lease,2528



collateral


...

,to_rating,count
0,BBB,4587
1,A,3914
2,B,3178
3,BB,3044
4,AA,1640
5,CCC,729
6,AAA,430
7,D,417



MACRO_STRESS_SCENARIOS — CATEGORICAL VALUES

scenario


,scenario,count
0,baseline,10
1,mild,10
2,adverse,10
3,severe,10
4,gfc_like,10
5,covid_like,10



sector


,sector,count
0,Financials,6
1,Real_Estate,6
2,Consumer,6
3,Industrials,6
4,Technology,6
5,Energy,6
6,Healthcare,6
7,Utilities,6
8,Retail,6
9,Telecom,6


In [19]:
# ---------------------------------------------------------------------------
# Date-range inspection
# ---------------------------------------------------------------------------

date_columns = {
    "loan_portfolio": [
        "origination_date",
        "maturity_date",
        "default_date",
    ],
    "credit_ratings": [],
    "macro_stress_scenarios": [],
    "portfolio_metrics": ["date"],
    "vintage_analysis": [],
}

for dataset_name, columns in date_columns.items():
    dataframe = datasets[dataset_name]

    print(f"\n{'=' * 80}")
    print(f"{dataset_name.upper()} — DATE RANGES")
    print(f"{'=' * 80}")

    for column in columns:
        dates = pd.to_datetime(
            dataframe[column],
            errors="coerce",
        )

        print(
            f"{column}: "
            f"{dates.min()} → {dates.max()} "
            f"| invalid/missing after parsing: {dates.isna().sum():,}"
        )


LOAN_PORTFOLIO — DATE RANGES
origination_date: 2015-01-01 00:00:00 → 2023-12-01 00:00:00 | invalid/missing after parsing: 0
maturity_date: 2016-01-01 00:00:00 → 2024-12-31 00:00:00 | invalid/missing after parsing: 0
default_date: 2015-02-01 00:00:00 → 2033-09-01 00:00:00 | invalid/missing after parsing: 43,050

CREDIT_RATINGS — DATE RANGES

MACRO_STRESS_SCENARIOS — DATE RANGES

PORTFOLIO_METRICS — DATE RANGES
date: 2015-01-01 00:00:00 → 2024-12-01 00:00:00 | invalid/missing after parsing: 0

VINTAGE_ANALYSIS — DATE RANGES


In [20]:
# ---------------------------------------------------------------------------
# Logical consistency checks
# ---------------------------------------------------------------------------

loan_checks = pd.DataFrame(
    {
        "check": [
            "defaulted = 1 with missing default_date",
            "defaulted = 0 with populated default_date",
            "maturity before origination",
            "default_date before origination",
            "default_date after maturity",
            "negative maturity_months",
            "invalid credit_score",
            "PD outside [0, 1]",
            "LGD outside [0, 1]",
            "recovery_rate outside [0, 1]",
            "negative EAD",
            "negative coupon_rate",
        ],
        "count": [
            (
                (loan_portfolio["defaulted"] == 1)
                & (loan_portfolio["default_date"].isna())
            ).sum(),
            (
                (loan_portfolio["defaulted"] == 0)
                & (loan_portfolio["default_date"].notna())
            ).sum(),
            (
                loan_portfolio["maturity_date"]
                < loan_portfolio["origination_date"]
            ).sum(),
            (
                loan_portfolio["default_date"]
                < loan_portfolio["origination_date"]
            ).sum(),
            (
                loan_portfolio["default_date"]
                > loan_portfolio["maturity_date"]
            ).sum(),
            (loan_portfolio["maturity_months"] < 0).sum(),
            (
                (loan_portfolio["credit_score"] < 0)
                | (loan_portfolio["credit_score"] > 1000)
            ).sum(),
            (
                (loan_portfolio["pd_annual"] < 0)
                | (loan_portfolio["pd_annual"] > 1)
            ).sum(),
            (
                (loan_portfolio["lgd"] < 0)
                | (loan_portfolio["lgd"] > 1)
            ).sum(),
            (
                (loan_portfolio["recovery_rate"] < 0)
                | (loan_portfolio["recovery_rate"] > 1)
            ).sum(),
            (loan_portfolio["ead"] < 0).sum(),
            (loan_portfolio["coupon_rate"] < 0).sum(),
        ],
    }
)

display(loan_checks)

,check,count
0,defaulted = 1 with missing default_date,0
1,defaulted = 0 with populated default_date,0
2,maturity before origination,0
3,default_date before origination,0
4,default_date after maturity,662
5,negative maturity_months,0
6,invalid credit_score,0
7,"PD outside [0, 1]",0
8,"LGD outside [0, 1]",0
9,"recovery_rate outside [0, 1]",0


In [21]:
# ---------------------------------------------------------------------------
# Recovery and loss consistency
# ---------------------------------------------------------------------------

defaulted_loans = loan_portfolio[
    loan_portfolio["defaulted"] == 1
].copy()

defaulted_loans["lgd_from_recovery"] = (
    1 - defaulted_loans["recovery_rate"]
)

defaulted_loans["loss_from_recovery"] = (
    defaulted_loans["ead"]
    * defaulted_loans["lgd_from_recovery"]
)

defaulted_loans["loss_difference"] = (
    defaulted_loans["loss_given_default"]
    - defaulted_loans["loss_from_recovery"]
)

recovery_check = {
    "defaulted_loans": len(defaulted_loans),
    "lgd_max_difference": (
        defaulted_loans["lgd"]
        - defaulted_loans["lgd_from_recovery"]
    ).abs().max(),
    "loss_max_difference": (
        defaulted_loans["loss_difference"]
        .abs()
        .max()
    ),
}

display(pd.Series(recovery_check))

defaulted_loans        6.950000e+03
lgd_max_difference     1.110223e-16
loss_max_difference    8.779000e-03
dtype: float64

In [22]:
# ---------------------------------------------------------------------------
# Expected Credit Loss consistency
# ---------------------------------------------------------------------------

loan_portfolio["ecl_calculated"] = (
    loan_portfolio["pd_annual"]
    * loan_portfolio["lgd"]
    * loan_portfolio["ead"]
)

loan_portfolio["ecl_difference"] = (
    loan_portfolio["ecl"]
    - loan_portfolio["ecl_calculated"]
)

ecl_check = loan_portfolio["ecl_difference"].abs()

print(f"Maximum absolute ECL difference: {ecl_check.max():,.6f}")
print(f"Mean absolute ECL difference:    {ecl_check.mean():,.6f}")
print(f"Median absolute ECL difference:  {ecl_check.median():,.6f}")

KeyError: 'ecl'

In [23]:
# ---------------------------------------------------------------------------
# Inspect loan portfolio column names
# ---------------------------------------------------------------------------

print("Loan portfolio columns:")

for column in loan_portfolio.columns:
    print(repr(column))

Loan portfolio columns:
'loan_id'
'origination_date'
'maturity_date'
'maturity_months'
'sector'
'loan_type'
'collateral'
'initial_rating'
'credit_score'
'ead'
'coupon_rate'
'leverage'
'interest_coverage'
'debt_to_equity'
'pd_annual'
'lgd'
'el'
'unexpected_loss'
'rwa'
'defaulted'
'default_date'
'survival_months'
'recovery_rate'
'loss_given_default'
'ecl_calculated'


In [24]:
# ---------------------------------------------------------------------------
# Find columns related to expected credit loss
# ---------------------------------------------------------------------------

ecl_candidates = [
    column
    for column in loan_portfolio.columns
    if "ecl" in column.lower()
    or "expected" in column.lower()
]

print("Potential ECL-related columns:")
print(ecl_candidates)

Potential ECL-related columns:
['unexpected_loss', 'ecl_calculated']


In [25]:
# ---------------------------------------------------------------------------
# Expected Credit Loss (ECL) consistency check
# ---------------------------------------------------------------------------

loan_portfolio["ecl_calculated"] = (
    loan_portfolio["pd_annual"]
    * loan_portfolio["lgd"]
    * loan_portfolio["ead"]
)

loan_portfolio["el_difference"] = (
    loan_portfolio["el"]
    - loan_portfolio["ecl_calculated"]
)

el_difference = loan_portfolio["el_difference"].abs()

print(f"Maximum absolute difference: {el_difference.max():,.6f}")
print(f"Mean absolute difference:    {el_difference.mean():,.6f}")
print(f"Median absolute difference:  {el_difference.median():,.6f}")

Maximum absolute difference: 68.962806
Mean absolute difference:    0.434541
Median absolute difference:  0.143740


In [26]:
# ---------------------------------------------------------------------------
# Inspect records with the largest EL differences
# ---------------------------------------------------------------------------

el_comparison = (
    loan_portfolio[
        [
            "loan_id",
            "pd_annual",
            "lgd",
            "ead",
            "el",
            "ecl_calculated",
            "el_difference",
        ]
    ]
    .copy()
)

el_comparison["absolute_difference"] = (
    el_comparison["el_difference"].abs()
)

display(
    el_comparison
    .sort_values("absolute_difference", ascending=False)
    .head(10)
)

,loan_id,pd_annual,lgd,ead,el,ecl_calculated,el_difference,absolute_difference
5698,L005699,0.010262,0.4219,3.611778e+08,1563663.90,1.563733e+06,-68.962806,68.962806
49123,L049124,0.042241,0.6235,1.785508e+08,4702581.39,4.702538e+06,43.150107,43.150107
18905,L018906,0.002679,0.6655,1.294463e+08,230746.26,2.307865e+05,-40.221876,40.221876
37293,L037294,0.000876,0.5822,1.533992e+08,78198.25,7.823469e+04,-36.436947,36.436947
879,L000880,0.066377,0.3958,2.111342e+08,5546886.56,5.546921e+06,-33.951019,33.951019
25748,L025749,0.000892,0.5212,1.293426e+08,60100.26,6.013272e+04,-32.458271,32.458271
13957,L013958,0.000867,0.5257,1.915292e+08,87324.55,8.729553e+04,29.016557,29.016557
26981,L026982,0.000853,0.6951,7.789424e+07,46211.53,4.618507e+04,26.455108,26.455108
3715,L003716,0.001161,0.7265,8.918024e+07,75194.85,7.522055e+04,-25.695222,25.695222
32780,L032781,0.042614,0.5735,9.244362e+07,2259215.92,2.259242e+06,-25.636607,25.636607


In [27]:
# ---------------------------------------------------------------------------
# Expected Loss relative-difference analysis
# ---------------------------------------------------------------------------

loan_portfolio["el_relative_difference"] = (
    loan_portfolio["el_difference"].abs()
    / loan_portfolio["el"].abs()
)

el_relative_difference = loan_portfolio[
    loan_portfolio["el"].ne(0)
]["el_relative_difference"]

el_validation = pd.Series(
    {
        "max_relative_difference_pct": (
            el_relative_difference.max() * 100
        ),
        "mean_relative_difference_pct": (
            el_relative_difference.mean() * 100
        ),
        "median_relative_difference_pct": (
            el_relative_difference.median() * 100
        ),
        "records_within_0_01_pct": (
            (el_relative_difference <= 0.0001).mean() * 100
        ),
        "records_within_0_1_pct": (
            (el_relative_difference <= 0.001).mean() * 100
        ),
    }
)

display(
    el_validation.to_frame("value")
)

,value
max_relative_difference_pct,0.321341
mean_relative_difference_pct,0.016026
median_relative_difference_pct,0.003936
records_within_0_01_pct,64.704000
records_within_0_1_pct,97.140000


In [28]:
# ---------------------------------------------------------------------------
# Expected Loss validation conclusion
# ---------------------------------------------------------------------------

tolerance = 0.001  # 0.1%

within_tolerance = (
    el_relative_difference <= tolerance
).mean()

print(
    f"{within_tolerance:.2%} of non-zero EL records "
    f"are within ±0.1% of the independently calculated value."
)

97.14% of non-zero EL records are within ±0.1% of the independently calculated value.


In [29]:
# ---------------------------------------------------------------------------
# LGD and recovery consistency check
# ---------------------------------------------------------------------------

defaulted_loans = loan_portfolio.loc[
    loan_portfolio["defaulted"].eq(1)
].copy()

defaulted_loans["lgd_calculated"] = (
    1 - defaulted_loans["recovery_rate"]
)

defaulted_loans["loss_calculated"] = (
    defaulted_loans["ead"]
    * defaulted_loans["lgd_calculated"]
)

defaulted_loans["lgd_difference"] = (
    defaulted_loans["lgd"]
    - defaulted_loans["lgd_calculated"]
)

defaulted_loans["loss_difference"] = (
    defaulted_loans["loss_given_default"]
    - defaulted_loans["loss_calculated"]
)

lgd_summary = pd.Series(
    {
        "defaulted_loans": len(defaulted_loans),
        "missing_recovery_rate": int(
            defaulted_loans["recovery_rate"].isna().sum()
        ),
        "missing_lgd": int(
            defaulted_loans["lgd"].isna().sum()
        ),
        "missing_loss_given_default": int(
            defaulted_loans["loss_given_default"].isna().sum()
        ),
        "max_abs_lgd_difference": (
            defaulted_loans["lgd_difference"].abs().max()
        ),
        "mean_abs_lgd_difference": (
            defaulted_loans["lgd_difference"].abs().mean()
        ),
        "max_abs_loss_difference": (
            defaulted_loans["loss_difference"].abs().max()
        ),
        "mean_abs_loss_difference": (
            defaulted_loans["loss_difference"].abs().mean()
        ),
    }
)

display(lgd_summary.to_frame("value"))

,value
defaulted_loans,6.950000e+03
missing_recovery_rate,0.000000e+00
missing_lgd,0.000000e+00
missing_loss_given_default,0.000000e+00
max_abs_lgd_difference,1.110223e-16
mean_abs_lgd_difference,2.695286e-17
max_abs_loss_difference,8.779000e-03
mean_abs_loss_difference,2.740278e-03


In [30]:
# ---------------------------------------------------------------------------
# Inspect largest LGD discrepancies
# ---------------------------------------------------------------------------

lgd_comparison = (
    defaulted_loans[
        [
            "loan_id",
            "ead",
            "lgd",
            "recovery_rate",
            "lgd_calculated",
            "lgd_difference",
            "loss_given_default",
            "loss_calculated",
            "loss_difference",
        ]
    ]
    .assign(
        absolute_lgd_difference=lambda x: x["lgd_difference"].abs(),
        absolute_loss_difference=lambda x: x["loss_difference"].abs(),
    )
    .sort_values(
        "absolute_lgd_difference",
        ascending=False,
    )
)

display(lgd_comparison.head(10))

,loan_id,ead,lgd,recovery_rate,lgd_calculated,lgd_difference,loss_given_default,loss_calculated,loss_difference,absolute_lgd_difference,absolute_loss_difference
49753,L049754,881311.14,0.6215,0.3785,0.6215,1.110223e-16,547734.88,5.477349e+05,0.006490,1.110223e-16,0.006490
16207,L016208,2023715.73,0.5108,0.4892,0.5108,1.110223e-16,1033714.00,1.033714e+06,0.005116,1.110223e-16,0.005116
49937,L049938,3268435.94,0.6143,0.3857,0.6143,-1.110223e-16,2007800.20,2.007800e+06,0.002058,1.110223e-16,0.002058
49930,L049931,491264.39,0.5074,0.4926,0.5074,-1.110223e-16,249267.55,2.492676e+05,-0.001486,1.110223e-16,0.001486
49926,L049927,1435830.16,0.7082,0.2918,0.7082,1.110223e-16,1016854.92,1.016855e+06,0.000688,1.110223e-16,0.000688
49919,L049920,2667411.02,0.5156,0.4844,0.5156,-1.110223e-16,1375317.12,1.375317e+06,-0.001912,1.110223e-16,0.001912
49858,L049859,1407708.37,0.7311,0.2689,0.7311,-1.110223e-16,1029175.59,1.029176e+06,0.000693,1.110223e-16,0.000693
49835,L049836,899474.45,0.6243,0.3757,0.6243,-1.110223e-16,561541.90,5.615419e+05,0.000865,1.110223e-16,0.000865
283,L000284,953144.40,0.6116,0.3884,0.6116,1.110223e-16,582943.12,5.829431e+05,0.004960,1.110223e-16,0.004960
259,L000260,361085.39,0.6821,0.3179,0.6821,1.110223e-16,246296.34,2.462963e+05,-0.004519,1.110223e-16,0.004519


In [31]:
# ---------------------------------------------------------------------------
# Default timeline consistency checks
# ---------------------------------------------------------------------------

timeline_checks = {
    "defaulted_with_missing_default_date": (
        (loan_portfolio["defaulted"].eq(1))
        & (loan_portfolio["default_date"].isna())
    ).sum(),

    "non_defaulted_with_default_date": (
        (loan_portfolio["defaulted"].eq(0))
        & (loan_portfolio["default_date"].notna())
    ).sum(),

    "maturity_before_origination": (
        loan_portfolio["maturity_date"]
        < loan_portfolio["origination_date"]
    ).sum(),

    "default_before_origination": (
        loan_portfolio["default_date"]
        < loan_portfolio["origination_date"]
    ).sum(),

    "default_after_maturity": (
        loan_portfolio["default_date"]
        > loan_portfolio["maturity_date"]
    ).sum(),
}

display(
    pd.Series(timeline_checks, name="count")
)

defaulted_with_missing_default_date      0
non_defaulted_with_default_date          0
maturity_before_origination              0
default_before_origination               0
default_after_maturity                 662
Name: count, dtype: int64

In [32]:
# ---------------------------------------------------------------------------
# Inspect defaults occurring after contractual maturity
# ---------------------------------------------------------------------------

post_maturity_defaults = (
    loan_portfolio.loc[
        loan_portfolio["default_date"]
        > loan_portfolio["maturity_date"],
        [
            "loan_id",
            "origination_date",
            "maturity_date",
            "default_date",
            "maturity_months",
            "survival_months",
            "defaulted",
            "initial_rating",
            "loan_type",
            "sector",
        ],
    ]
    .sort_values("default_date")
)

print(
    f"Defaults after maturity: "
    f"{len(post_maturity_defaults):,}"
)

display(post_maturity_defaults.head(20))

Defaults after maturity: 662


,loan_id,origination_date,maturity_date,default_date,maturity_months,survival_months,defaulted,initial_rating,loan_type,sector
38279,L038280,2022-09-01,2024-12-31,2025-01-01,60,28,1,CCC,term_loan,Telecom
1539,L001540,2022-05-01,2024-12-31,2025-01-01,36,32,1,B,revolving,Energy
19554,L019555,2022-01-01,2024-12-31,2025-01-01,48,36,1,B,bond,Technology
20172,L020173,2023-05-01,2024-12-31,2025-01-01,36,20,1,CCC,term_loan,Healthcare
9104,L009105,2022-08-01,2024-12-31,2025-01-01,84,29,1,CCC,term_loan,Telecom
21458,L021459,2023-01-01,2024-12-31,2025-01-01,48,24,1,CCC,term_loan,Industrials
1285,L001286,2021-02-01,2024-12-31,2025-01-01,84,47,1,CCC,revolving,Technology
30096,L030097,2020-04-01,2024-12-31,2025-01-01,120,57,1,CCC,bond,Healthcare
8159,L008160,2020-12-01,2024-12-31,2025-01-01,60,49,1,CCC,bond,Telecom
26831,L026832,2020-08-01,2024-12-31,2025-01-01,120,53,1,CCC,term_loan,Technology


In [35]:
# ---------------------------------------------------------------------------
# Calculate post-maturity default delay
# ---------------------------------------------------------------------------

post_maturity_defaults = post_maturity_defaults.copy()

post_maturity_defaults["maturity_date"] = pd.to_datetime(
    post_maturity_defaults["maturity_date"],
    errors="coerce",
)

post_maturity_defaults["default_date"] = pd.to_datetime(
    post_maturity_defaults["default_date"],
    errors="coerce",
)

post_maturity_defaults["days_after_maturity"] = (
    post_maturity_defaults["default_date"]
    - post_maturity_defaults["maturity_date"]
).dt.days

print(
    f"Post-maturity defaults: "
    f"{len(post_maturity_defaults):,}"
)

display(
    post_maturity_defaults["days_after_maturity"]
    .value_counts()
    .sort_index()
    .head(20)
)

Post-maturity defaults: 662


days_after_maturity
1      27
32     21
60     21
91     34
121    26
152    13
182    24
213    20
244    21
274    19
305    21
335    14
366    10
397    15
425    21
456    15
486    16
517    15
547    13
578    11
Name: count, dtype: int64

In [36]:
# ---------------------------------------------------------------------------
# Summary of post-maturity default delays
# ---------------------------------------------------------------------------

delay_summary = (
    post_maturity_defaults["days_after_maturity"]
    .describe()
)

display(delay_summary)

count     662.000000
mean      656.345921
std       604.858348
min         1.000000
25%       189.750000
50%       486.000000
75%       912.000000
max      3166.000000
Name: days_after_maturity, dtype: float64

In [37]:
# ---------------------------------------------------------------------------
# Share of defaults occurring exactly one day after maturity
# ---------------------------------------------------------------------------

one_day_default_rate = (
    post_maturity_defaults["days_after_maturity"]
    .eq(1)
    .mean()
)

print(
    f"Exactly one day after maturity: "
    f"{one_day_default_rate:.2%}"
)

Exactly one day after maturity: 4.08%


In [38]:
# ---------------------------------------------------------------------------
# Maturity-date distribution for post-maturity defaults
# ---------------------------------------------------------------------------

maturity_distribution = (
    post_maturity_defaults["maturity_date"]
    .value_counts()
    .sort_index()
)

display(maturity_distribution)

maturity_date
2024-12-31    662
Name: count, dtype: int64

### Post-Maturity Default Finding

A total of **662 loans** have `default_date > maturity_date`.

All 662 observations have the same contractual maturity date:

**2024-12-31**

The default delay ranges from **1 to 3,166 days**, with a median of **486 days**, and only **4.08%** occur exactly one day after maturity.

**Interpretation:** This is not a simple one-day or random date-entry error. The concentration of all such observations at 2024-12-31 suggests a structural boundary in the synthetic dataset. These observations will be retained but flagged for temporal analysis. They will not be silently corrected or removed.

**Modelling implication:** Default timing, survival analysis, vintage analysis, and any out-of-time validation must account for the dataset's observation boundary.

In [39]:
# ---------------------------------------------------------------------------
# Portfolio date boundaries
# ---------------------------------------------------------------------------

date_boundaries = pd.Series(
    {
        "latest_origination_date": loan_portfolio["origination_date"].max(),
        "latest_maturity_date": loan_portfolio["maturity_date"].max(),
        "latest_default_date": loan_portfolio["default_date"].max(),
    }
)

display(date_boundaries.to_frame("date"))

,date
latest_origination_date,2023-12-01
latest_maturity_date,2024-12-31
latest_default_date,2033-09-01


In [40]:
# ---------------------------------------------------------------------------
# Survival-period consistency check
# ---------------------------------------------------------------------------

loan_dates = loan_portfolio[
    loan_portfolio["defaulted"].eq(1)
].copy()

loan_dates["origination_date"] = pd.to_datetime(
    loan_dates["origination_date"],
    errors="coerce",
)

loan_dates["default_date"] = pd.to_datetime(
    loan_dates["default_date"],
    errors="coerce",
)

loan_dates["observed_months"] = (
    (
        loan_dates["default_date"]
        - loan_dates["origination_date"]
    ).dt.days
    / 30.4375
)

loan_dates["survival_difference"] = (
    loan_dates["survival_months"]
    - loan_dates["observed_months"]
)

display(
    loan_dates[
        [
            "loan_id",
            "origination_date",
            "default_date",
            "survival_months",
            "observed_months",
            "survival_difference",
        ]
    ].head(20)
)

,loan_id,origination_date,default_date,survival_months,observed_months,survival_difference
7,L000008,2018-12-01,2020-04-01,16,16.000000,0.000000
9,L000010,2018-06-01,2020-04-01,22,22.012320,-0.012320
11,L000012,2021-07-01,2021-08-01,1,1.018480,-0.018480
19,L000020,2019-04-01,2020-06-01,14,14.028747,-0.028747
33,L000034,2020-09-01,2021-10-01,13,12.977413,0.022587
35,L000036,2016-02-01,2018-08-01,30,29.963039,0.036961
37,L000038,2020-04-01,2022-08-01,28,27.991786,0.008214
48,L000049,2023-09-01,2027-04-01,43,42.973306,0.026694
49,L000050,2019-02-01,2020-04-01,14,13.963039,0.036961
51,L000052,2018-12-01,2020-06-01,18,18.004107,-0.004107


In [41]:
# ---------------------------------------------------------------------------
# Survival-period discrepancy summary
# ---------------------------------------------------------------------------

survival_difference_summary = (
    loan_dates["survival_difference"]
    .describe()
)

display(survival_difference_summary.to_frame("value"))

,value
count,6950.000000
mean,0.000484
std,0.028849
min,-0.096509
25%,-0.020534
50%,0.000000
75%,0.018480
max,0.096509


### Survival-Months Validation — Conclusion

The supplied `survival_months` field closely reproduces the elapsed time between
`origination_date` and `default_date`.

The mean difference is only **0.000484 months**, with a maximum absolute
difference of **0.096509 months** (approximately 2.9 days).

**Conclusion:** `survival_months` is an outcome-derived variable and will be
excluded from the origination-time PD feature set to prevent target leakage.

It may be retained for survival, vintage, and default-timing analysis.

In [42]:
# ---------------------------------------------------------------------------
# PD distribution and default relationship
# ---------------------------------------------------------------------------

pd_summary = (
    loan_portfolio
    .groupby("defaulted")["pd_annual"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        min="min",
        max="max",
    )
    .rename(index={0: "Non-defaulted", 1: "Defaulted"})
)

display(pd_summary)

,count,mean,median,min,max
defaulted,,,,,
Non-defaulted,43050,0.016502,0.002713,0.000158,0.242523
Defaulted,6950,0.059374,0.039728,0.000163,0.241140


In [43]:
# ---------------------------------------------------------------------------
# PD relationship with observed default outcome
# ---------------------------------------------------------------------------

pd_by_rating = (
    loan_portfolio
    .groupby("initial_rating")
    .agg(
        loans=("loan_id", "count"),
        default_rate=("defaulted", "mean"),
        average_pd=("pd_annual", "mean"),
    )
    .sort_index()
)

pd_by_rating["default_rate_pct"] = (
    pd_by_rating["default_rate"] * 100
)

display(pd_by_rating)

,loans,default_rate,average_pd,default_rate_pct
initial_rating,,,,
A,7571,0.063928,0.000918,6.392815
AA,3944,0.063895,0.000454,6.389452
AAA,1529,0.052322,0.000180,5.232178
B,7559,0.218283,0.042275,21.828284
BB,10918,0.111742,0.011144,11.174208
BBB,14001,0.070424,0.002765,7.042354
CCC,4478,0.508709,0.141607,50.870925


In [44]:
# ---------------------------------------------------------------------------
# LGD distribution by default status
# ---------------------------------------------------------------------------

lgd_summary = (
    loan_portfolio
    .groupby("defaulted")["lgd"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        min="min",
        max="max",
    )
    .rename(index={0: "Non-defaulted", 1: "Defaulted"})
)

display(lgd_summary)

,count,mean,median,min,max
defaulted,,,,,
Non-defaulted,43050,0.546690,0.5449,0.0918,0.98
Defaulted,6950,0.545438,0.5439,0.1422,0.98


In [45]:
# ---------------------------------------------------------------------------
# LGD and recovery relationship
# ---------------------------------------------------------------------------

lgd_recovery_check = (
    loan_portfolio.loc[
        loan_portfolio["defaulted"].eq(1),
        ["lgd", "recovery_rate"],
    ]
    .copy()
)

lgd_recovery_check["lgd_from_recovery"] = (
    1 - lgd_recovery_check["recovery_rate"]
)

lgd_recovery_check["difference"] = (
    lgd_recovery_check["lgd"]
    - lgd_recovery_check["lgd_from_recovery"]
)

display(
    lgd_recovery_check["difference"]
    .abs()
    .describe()
    .to_frame("absolute_difference")
)

,absolute_difference
count,6.950000e+03
mean,2.695286e-17
std,4.095676e-17
min,0.000000e+00
25%,0.000000e+00
50%,0.000000e+00
75%,5.551115e-17
max,1.110223e-16


### LGD / Recovery Validation — Finding

For all 6,950 defaulted loans, the supplied `lgd` field is mathematically
consistent with the supplied recovery rate:

**LGD = 1 − Recovery Rate**

The maximum absolute difference is only **1.11 × 10⁻¹⁶**, which is effectively
zero and attributable to floating-point representation.

**Conclusion:** No data correction is required for the LGD/recovery relationship.

In [46]:
# ---------------------------------------------------------------------------
# LGD relationship with key risk characteristics
# ---------------------------------------------------------------------------

lgd_by_rating = (
    loan_portfolio
    .groupby("initial_rating")
    .agg(
        loans=("loan_id", "count"),
        average_lgd=("lgd", "mean"),
        average_recovery=("recovery_rate", "mean"),
    )
    .sort_index()
)

display(lgd_by_rating)

,loans,average_lgd,average_recovery
initial_rating,,,
A,7571,0.547009,0.446895
AA,3944,0.545865,0.455832
AAA,1529,0.540512,0.449943
B,7559,0.545331,0.456157
BB,10918,0.546226,0.451111
BBB,14001,0.547130,0.456974
CCC,4478,0.549094,0.455862


In [47]:
# ---------------------------------------------------------------------------
# LGD relationship with collateral
# ---------------------------------------------------------------------------

lgd_by_collateral = (
    loan_portfolio
    .groupby("collateral")
    .agg(
        loans=("loan_id", "count"),
        average_lgd=("lgd", "mean"),
        average_recovery=("recovery_rate", "mean"),
    )
    .sort_values("average_lgd")
)

display(lgd_by_collateral)

,loans,average_lgd,average_recovery
collateral,,,
secured,22657,0.452777,0.548896
partially_secured,9945,0.573737,0.429792
unsecured,17398,0.653030,0.344680


In [48]:
# ---------------------------------------------------------------------------
# LGD relationship with loan type
# ---------------------------------------------------------------------------

lgd_by_loan_type = (
    loan_portfolio
    .groupby("loan_type")
    .agg(
        loans=("loan_id", "count"),
        average_lgd=("lgd", "mean"),
        average_recovery=("recovery_rate", "mean"),
    )
    .sort_values("average_lgd")
)

display(lgd_by_loan_type)

,loans,average_lgd,average_recovery
loan_type,,,
revolving,10158,0.545422,0.461331
mortgage,12262,0.545685,0.450482
term_loan,17594,0.546548,0.455156
bond,7458,0.547528,0.452339
lease,2528,0.551738,0.449909


### LGD by Loan Type — Finding

Loan type shows only a small variation in average LGD across the synthetic
portfolio.

Average LGD ranges from approximately **54.54% for revolving loans** to
**55.17% for leases**, a spread of only about **0.63 percentage points**.

**Conclusion:** Loan type does not appear to be a major driver of LGD in this
dataset based on univariate analysis. It may be retained as a candidate
feature for multivariate testing, but no strong standalone relationship is
observed.

In [49]:
# ---------------------------------------------------------------------------
# EAD distribution and range checks
# ---------------------------------------------------------------------------

ead_summary = loan_portfolio["ead"].describe()

display(ead_summary.to_frame("value"))

,value
count,5.000000e+04
mean,3.298605e+06
std,6.661943e+06
min,5.000000e+04
25%,5.482685e+05
50%,1.389712e+06
75%,3.425439e+06
max,3.611778e+08


In [50]:
# ---------------------------------------------------------------------------
# EAD quality checks
# ---------------------------------------------------------------------------

ead_checks = pd.Series(
    {
        "missing_ead": int(loan_portfolio["ead"].isna().sum()),
        "zero_ead": int(loan_portfolio["ead"].eq(0).sum()),
        "negative_ead": int(loan_portfolio["ead"].lt(0).sum()),
        "positive_ead": int(loan_portfolio["ead"].gt(0).sum()),
    },
    name="count",
)

display(ead_checks.to_frame())

,count
missing_ead,0
zero_ead,0
negative_ead,0
positive_ead,50000


In [51]:
# ---------------------------------------------------------------------------
# EAD by loan type
# ---------------------------------------------------------------------------

ead_by_loan_type = (
    loan_portfolio
    .groupby("loan_type")
    .agg(
        loans=("loan_id", "count"),
        average_ead=("ead", "mean"),
        median_ead=("ead", "median"),
        min_ead=("ead", "min"),
        max_ead=("ead", "max"),
    )
    .sort_values("average_ead", ascending=False)
)

display(ead_by_loan_type)

,loans,average_ead,median_ead,min_ead,max_ead
loan_type,,,,,
term_loan,17594,3.396963e+06,1399367.140,50000.0,2.111342e+08
mortgage,12262,3.276356e+06,1399330.970,50000.0,1.533992e+08
bond,7458,3.274290e+06,1388630.435,50000.0,3.611778e+08
lease,2528,3.264413e+06,1401779.330,50000.0,1.141715e+08
revolving,10158,3.181463e+06,1362153.640,50000.0,9.652479e+07


### EAD by Loan Type — Finding

EAD distributions are broadly similar across loan types. Median exposure
ranges from approximately **1.36M to 1.40M**, while mean exposure is higher
because of a right-skewed distribution.

The largest observed exposure is approximately **361.18M**, belonging to a
bond exposure.

**Conclusion:** No immediate data-quality error is identified from the EAD
distribution. However, the extreme upper tail requires concentration and
outlier analysis before modelling. Extreme exposures will not be removed or
capped without a documented modelling justification.

In [52]:
largest_exposures = (
    loan_portfolio[
        [
            "loan_id",
            "ead",
            "loan_type",
            "sector",
            "initial_rating",
            "collateral",
            "defaulted",
        ]
    ]
    .sort_values("ead", ascending=False)
)

display(largest_exposures.head(20))

,loan_id,ead,loan_type,sector,initial_rating,collateral,defaulted
5698,L005699,3.611778e+08,bond,Real_Estate,BB,secured,0
879,L000880,2.111342e+08,term_loan,Telecom,B,secured,0
13957,L013958,1.915292e+08,term_loan,Financials,A,partially_secured,0
49123,L049124,1.785508e+08,term_loan,Industrials,B,unsecured,1
37293,L037294,1.533992e+08,mortgage,Telecom,A,secured,0
21176,L021177,1.481146e+08,mortgage,Utilities,BB,unsecured,0
9242,L009243,1.445477e+08,term_loan,Real_Estate,BBB,unsecured,0
37486,L037487,1.375627e+08,term_loan,Financials,BBB,secured,0
2208,L002209,1.357299e+08,mortgage,Utilities,BBB,secured,0
18905,L018906,1.294463e+08,term_loan,Financials,BBB,unsecured,0


In [53]:
# ---------------------------------------------------------------------------
# EAD concentration analysis
# ---------------------------------------------------------------------------

total_ead = loan_portfolio["ead"].sum()

largest_10_ead = (
    loan_portfolio
    .nlargest(10, "ead")["ead"]
    .sum()
)

largest_20_ead = (
    loan_portfolio
    .nlargest(20, "ead")["ead"]
    .sum()
)

ead_concentration = pd.Series(
    {
        "total_portfolio_ead": total_ead,
        "top_10_ead": largest_10_ead,
        "top_10_share_pct": (
            largest_10_ead / total_ead * 100
        ),
        "top_20_ead": largest_20_ead,
        "top_20_share_pct": (
            largest_20_ead / total_ead * 100
        ),
    }
)

display(ead_concentration.to_frame("value"))

,value
total_portfolio_ead,1.649302e+11
top_10_ead,1.791192e+09
top_10_share_pct,1.086030e+00
top_20_ead,2.940880e+09
top_20_share_pct,1.783106e+00


### EAD Concentration — Finding

The portfolio contains total EAD of approximately **£164.93 billion**.

The 10 largest exposures represent approximately **1.09%** of total portfolio
EAD, while the 20 largest exposures represent approximately **1.78%**.

Although several individual exposures are very large, they do not dominate the
overall portfolio.

**Conclusion:** Extreme EAD observations should be retained. They represent
potentially important large exposures rather than obvious data-quality errors.
Further sector/rating concentration analysis will be used to assess portfolio
risk.

In [54]:
# ---------------------------------------------------------------------------
# EAD concentration by sector
# ---------------------------------------------------------------------------

ead_by_sector = (
    loan_portfolio
    .groupby("sector")
    .agg(
        loans=("loan_id", "count"),
        total_ead=("ead", "sum"),
        average_ead=("ead", "mean"),
        median_ead=("ead", "median"),
    )
    .sort_values("total_ead", ascending=False)
)

ead_by_sector["ead_share_pct"] = (
    ead_by_sector["total_ead"]
    / total_ead
    * 100
)

display(ead_by_sector)

,loans,total_ead,average_ead,median_ead,ead_share_pct
sector,,,,,
Financials,5023,3.016669e+10,6.005713e+06,3055114.700,18.290579
Real_Estate,4862,2.530543e+10,5.204737e+06,2582803.160,15.343113
Utilities,4959,2.206999e+10,4.450491e+06,2138756.350,13.381408
Energy,5132,2.093550e+10,4.079403e+06,2053906.250,12.693546
Telecom,5077,1.891039e+10,3.724716e+06,1822229.330,11.465687
Industrials,5012,1.571215e+10,3.134907e+06,1494056.875,9.526545
Technology,4959,1.277305e+10,2.575731e+06,1216800.280,7.744517
Healthcare,4982,1.004774e+10,2.016809e+06,959219.380,6.092118
Consumer,5043,5.049910e+09,1.001370e+06,505612.340,3.061846


### EAD Concentration by Sector — Finding

Portfolio exposure is materially concentrated by sector.

The five largest sectors by EAD are:

- Financials: **18.29%**
- Real Estate: **15.34%**
- Utilities: **13.38%**
- Energy: **12.69%**
- Telecom: **11.47%**

Together, these five sectors represent approximately **71.17% of total
portfolio EAD**.

**Interpretation:** Although the largest individual loans do not dominate the
portfolio, sector-level concentration is significant.

**Risk implication:** Sector exposure should be evaluated together with PD and
LGD. High EAD alone does not mean high credit risk.

A later portfolio-risk stage will calculate sector-level Expected Credit Loss
and stress impacts.

In [56]:
# ---------------------------------------------------------------------------
# Final audit summary
# ---------------------------------------------------------------------------

audit_findings = {
    "loan_records": len(loan_portfolio),
    "defaulted_loans": int(loan_portfolio["defaulted"].sum()),
    "default_rate_pct": round(
        loan_portfolio["defaulted"].mean() * 100,
        2,
    ),
    "duplicate_loan_ids": int(
        loan_portfolio["loan_id"].duplicated().sum()
    ),
    "missing_ead": int(
        loan_portfolio["ead"].isna().sum()
    ),
    "negative_ead": int(
        loan_portfolio["ead"].lt(0).sum()
    ),
    "post_maturity_defaults": int(
        (
            loan_portfolio["default_date"]
            > loan_portfolio["maturity_date"]
        ).sum()
    ),
    "total_portfolio_ead": loan_portfolio["ead"].sum(),
    "top_10_ead_share_pct": round(
        largest_10_ead / total_ead * 100,
        2,
    ),
    "top_20_ead_share_pct": round(
        largest_20_ead / total_ead * 100,
        2,
    ),
}

audit_summary = pd.Series(
    audit_findings,
    name="value",
)

display(audit_summary.to_frame())

,value
loan_records,5.000000e+04
defaulted_loans,6.950000e+03
default_rate_pct,1.390000e+01
duplicate_loan_ids,0.000000e+00
missing_ead,0.000000e+00
negative_ead,0.000000e+00
post_maturity_defaults,6.620000e+02
total_portfolio_ead,1.649302e+11
top_10_ead_share_pct,1.090000e+00
top_20_ead_share_pct,1.780000e+00


# Audit Conclusion

The source portfolio contains 50,000 unique loans and 6,950 observed defaults.

The audit identified:

- No duplicate loan IDs.
- No missing, zero, or negative EAD values.
- No inconsistent default-status/default-date relationships.
- No maturity dates preceding origination dates.
- Strong mathematical consistency between LGD and recovery rate.
- Strong consistency between supplied expected loss and independently calculated PD × LGD × EAD.
- `survival_months` is effectively derived from origination-to-default timing and will therefore be excluded from the origination-time PD feature set.
- `pd_annual` will be treated as a benchmark/reference measure rather than a primary model feature.
- `default_date`, `survival_months`, recovery and loss measures are outcome/post-outcome variables and require exclusion from the origination PD feature set.
- 662 loans have default dates after contractual maturity. These observations are retained but flagged for temporal analysis.
- Portfolio EAD is right-skewed with several very large exposures; these will not be removed without modelling justification.
- Sector concentration is material, with the five largest sectors representing approximately 71.17% of portfolio EAD.

**Overall decision:** The dataset is suitable for further BankSense 2.0 development, provided temporal anomalies, leakage, and synthetic-data limitations are explicitly documented and handled.